In [ ]:
# Installation des dépendances nécessaires :
# !pip install python-binance pandas yfinance requests matplotlib seaborn statsmodels scipy deap scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
import random
from datetime import datetime
from itertools import combinations

# Bibliothèques d'analyse statistique et économétrique
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from statsmodels.tsa.ar_model import AutoReg
from sklearn.model_selection import TimeSeriesSplit

# Bibliothèques pour l'optimisation par algorithme génétique
from deap import base, creator, tools, algorithms

from binance.client import Client
from binance.enums import *

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION ET PARAMÈTRES
# ==========================================

api_key = 'Votre cle API'
api_secret = 'Votre code secret API'
client = Client(api_key, api_secret)

# Liste des cryptos à analyser
SYMBOLS = [
    "FIROUSDT", "DEXEUSDT", "ERNUSDT", "HARDUSDT", "STMXUSDT",
    "DIAUSDT", "PNTUSDT", "UMAUSDT", "SFPUSDT", "MLNUSDT",
    "DATAUSDT", "PONDUSDT", "ANKRUSDT", "DENTUSDT", "LINAUSDT",
    "SUNUSDT", "ARDRUSDT", "DOCKUSDT", "CTSIUSDT", "KMDUSDT",
    "OGNUSDT", "SCUSDT", "BELUSDT", "CELOUSDT", "IOTXUSDT",
    "LTOUSDT", "OXTUSDT", "QTUMUSDT", "ZENUSDT", "BANDUSDT"
]

START_DATE = "2024-12-12"
END_DATE = "2025-12-12"
INTERVAL = Client.KLINE_INTERVAL_1HOUR
FILE_NAME = "crypto_basket.csv"

# Paramètres de stratégie
SIGNIFICANCE_LEVEL = 0.05
FEE = 0.001  # Frais de transaction (0.1%)

# ==========================================
# 2. COLLECTE ET PRÉPARATION DES DONNÉES
# ==========================================

def get_close_data(symbol):
    """Télécharge les prix de clôture depuis l'API Binance."""
    try:
        print(f"Téléchargement de {symbol}...")
        klines = client.get_historical_klines(symbol, INTERVAL, START_DATE, END_DATE)
        data = pd.DataFrame(klines, columns=['Open time', 'Close'] + ['_']*10)
        data['Open time'] = pd.to_datetime(data['Open time'], unit='ms')
        data.set_index('Open time', inplace=True)
        data['Close'] = data['Close'].astype(float)
        data.rename(columns={'Close': symbol}, inplace=True)
        return data[[symbol]]
    except Exception as e:
        print(f"Erreur sur {symbol}: {e}")
        return None

# Exécution du téléchargement et sauvegarde
all_dfs = []
for sym in SYMBOLS:
    df_temp = get_close_data(sym)
    if df_temp is not None:
        all_dfs.append(df_temp)
    time.sleep(0.1)

if all_dfs:
    df_basket = pd.concat(all_dfs, axis=1)
    df_basket.to_csv(FILE_NAME)

# Nettoyage et transformation logarithmique
df = pd.read_csv(FILE_NAME, index_col=0, parse_dates=True)
df = df.replace([np.inf, -np.inf, 0], np.nan).dropna(axis=1, how='all')
df_log = np.log(df)

# ==========================================
# 3. ANALYSE DE COINTÉGRATION ET CLASSEMENT
# ==========================================

assets = df_log.columns.tolist()
pairs = list(combinations(assets, 2))
results = []
all_pvalues = []

print(f"\nAnalyse de cointégration sur {len(pairs)} paires...")

for asset1, asset2 in pairs:
    series1, series2 = df_log[asset1], df_log[asset2]
    valid_data = pd.concat([series1, series2], axis=1).dropna()
    
    if len(valid_data) < 100: continue
    
    s1, s2 = valid_data[asset1], valid_data[asset2]

    try:
        # Test de cointégration d'Engle-Granger
        score, pvalue, _ = coint(s1, s2)
        all_pvalues.append(pvalue)
        
        if pvalue < SIGNIFICANCE_LEVEL:
            # Calcul du Hedge Ratio (Beta) via OLS
            X = sm.add_constant(s2)
            model = sm.OLS(s1, X).fit()
            residuals = model.resid
            beta = model.params[asset2]
            
            # Analyse du Modèle à Correction d'Erreur (ECM)
            ect = residuals.shift(1).dropna()
            d_y1, d_y2 = s1.diff().dropna(), s2.diff().dropna()
            common_idx = ect.index.intersection(d_y1.index).intersection(d_y2.index)
            
            X_ecm = sm.add_constant(ect.loc[common_idx])
            mod_ecm1 = sm.OLS(d_y1.loc[common_idx], X_ecm).fit()
            mod_ecm2 = sm.OLS(d_y2.loc[common_idx], X_ecm).fit()
            
            l1, p1 = mod_ecm1.params[0], mod_ecm1.pvalues[0]
            l2, p2 = mod_ecm2.params[0], mod_ecm2.pvalues[0]
            
            # Détection des anomalies (Sauts/Jumps)
            jumps = (np.abs(residuals) > (3 * residuals.std())).sum()
            
            # Système de notation (Ranking)
            is_double_rev = (p1 < 0.05 and l1 < 0) and (p2 < 0.05 and l2 > 0)
            is_stable = jumps < 20
            
            if is_double_rev and is_stable: rank, status = 1, "lvl 1 : Très bien"
            elif is_double_rev: rank, status = 2, "lvl 2 : Volatilité élevée"
            elif (p1 < 0.05 and l1 < 0) or (p2 < 0.05 and l2 > 0): rank, status = 3, "lvl 3 : Correction unilatérale"
            else: rank, status = 4, "lvl 4 : Insuffisant"

            results.append({
                "Rank": rank, "Status": status, "Pair": f"{asset1}/{asset2}",
                "Coint_Pvalue": round(pvalue, 5), "Beta": round(beta, 4),
                "Jumps": jumps, "Lambda1": round(l1, 4), "Lambda2": round(l2, 4)
            })
    except: continue

df_res = pd.DataFrame(results).sort_values(["Rank", "Coint_Pvalue"])
print("\nTop 15 des meilleures paires identifiées :")
print(df_res.head(15))

# ==========================================
# 4. TEST DE ROBUSTESSE (ROLLING TEST)
# ==========================================

TOP_N = 3
WINDOW_ROLL = 24 * 60 # 60 jours
STEP_ROLL = 24        # Pas quotidien

if not df_res.empty:
    candidates = df_res[df_res['Rank'] == 1].head(TOP_N)
    for _, row in candidates.iterrows():
        p_name = row['Pair']
        a1, a2 = p_name.split('/')
        print(f"\nAnalyse de robustesse glissante pour {p_name}...")
        
        s1, s2 = df_log[a1], df_log[a2]
        r_dates, r_pvals = [], []
        
        for t in range(WINDOW_ROLL, len(df_log), STEP_ROLL):
            sub1, sub2 = s1.iloc[t-WINDOW_ROLL:t], s2.iloc[t-WINDOW_ROLL:t]
            try:
                _, p, _ = coint(sub1, sub2)
                r_dates.append(df_log.index[t])
                r_pvals.append(p)
            except: continue
        
        plt.figure(figsize=(10, 3))
        plt.plot(r_dates, r_pvals, label='P-Value Glissante')
        plt.axhline(0.05, color='red', linestyle='--')
        plt.title(f"Stabilité de la cointégration : {p_name}")
        plt.legend()
        plt.show()

# ==========================================
# 5. OPTIMISATION PAR ALGORITHME GÉNÉTIQUE
# ==========================================

# Sélection de la meilleure paire pour l'optimisation
target_pair = df_res.iloc[0]['Pair']
ASSET_1, ASSET_2 = target_pair.split('/')
pair_data = df_log[[ASSET_1, ASSET_2]].dropna()
split_idx = int(len(pair_data) * 0.7)

# Calcul du Hedge Ratio sur l'échantillon d'entraînement
X_train = sm.add_constant(pair_data[ASSET_2].iloc[:split_idx])
h_ratio = sm.OLS(pair_data[ASSET_1].iloc[:split_idx], X_train).fit().params[ASSET_2]
pair_data['Spread'] = pair_data[ASSET_1] - (h_ratio * pair_data[ASSET_2])

def backtest_logic(individual, data_segment):
    """Fonction de simulation pour l'évaluation de la stratégie."""
    window, entry, stop, exit_val = int(individual[0]), individual[1], individual[2], individual[3]
    if stop <= entry or window < 5: return -999,
    
    roll_mean = data_segment.rolling(window).mean()
    roll_std = data_segment.rolling(window).std()
    z = ((data_segment - roll_mean) / roll_std).values
    prices = data_segment.values
    
    returns, pos = [], 0
    for i in range(window, len(z)):
        pnl = (prices[i] - prices[i-1]) * pos if pos != 0 else 0
        returns.append(pnl)
        
        if pos == 0:
            if z[i] > entry: pos = -1
            elif z[i] < -entry: pos = 1
        elif pos == -1:
            if z[i] > stop or z[i] < exit_val: pos = 0
        elif pos == 1:
            if z[i] < -stop or z[i] > -exit_val: pos = 0
            
    if not returns or np.std(returns) == 0: return -999,
    return (np.mean(returns) / np.std(returns)) * np.sqrt(24 * 365),

# Configuration de l'algorithme génétique (DEAP)
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_win", random.randint, 20, 400)
toolbox.register("attr_ent", random.uniform, 1.0, 3.5)
toolbox.register("attr_stp", random.uniform, 4.0, 9.0)
toolbox.register("attr_ext", random.uniform, -0.5, 0.5)
toolbox.register("individual", tools.initCycle, creator.Individual, (toolbox.attr_win, toolbox.attr_ent, toolbox.attr_stp, toolbox.attr_ext), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", lambda ind: backtest_logic(ind, pair_data['Spread'].iloc[:split_idx]))
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

print(f"\nOptimisation de la stratégie sur {ASSET_1}/{ASSET_2}...")
pop = toolbox.population(n=100)
hof = tools.HallOfFame(1)
algorithms.eaSimple(pop, toolbox, cxpb=0.6, mutpb=0.2, ngen=20, verbose=False, halloffame=hof)

# ==========================================
# 6. ANALYSE DES RÉSULTATS FINAUX
# ==========================================

best_params = hof[0]
print(f"\n--- RÉSULTATS FINAUX ({ASSET_1}/{ASSET_2}) ---")
print(f"Fenêtre optimale : {int(best_params[0])}h")
print(f"Seuils : Entrée {best_params[1]:.2f} | Stop {best_params[2]:.2f} | Sortie {best_params[3]:.2f}")

# Calcul des performances sur l'échantillon Test (Out-of-Sample)
sharpe_test = backtest_logic(best_params, pair_data['Spread'].iloc[split_idx:])[0]
print(f"Ratio de Sharpe (Validation Test) : {sharpe_test:.4f}")

# Visualisation des prix historiques
plt.figure(figsize=(12, 5))
plt.plot(df[ASSET_1], label=ASSET_1, alpha=0.8)
plt.plot(df[ASSET_2], label=ASSET_2, alpha=0.8)
plt.title(f"Evolution des prix : {ASSET_1} et {ASSET_2}")
plt.legend()
plt.show()